In [1]:
import sys
import pickle
import random
from typing import Optional, Union

from poke_env import RandomPlayer
from poke_env.data import GenData
from poke_env import AccountConfiguration
from poke_env.environment import SinglesEnv
from poke_env.environment.env import _EnvPlayer
from poke_env.battle import AbstractBattle, Battle
from poke_env.teambuilder import Teambuilder
from poke_env.player import Player, RandomPlayer



import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

from collections import defaultdict
from typing import Any, Dict, Optional

from gymnasium.spaces import Box, Discrete, Space, MultiDiscrete
import torch
import torch.nn as nn
from torch import multiprocessing
from tensordict import TensorDict, TensorDictBase
from tensordict.nn import TensorDictModule
from tensordict.nn.distributions import NormalParamExtractor


from torchrl.collectors import SyncDataCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.envs import (Compose, DoubleToFloat, ObservationNorm, StepCounter,
                          TransformedEnv)


#Custom Env pytorch tutorial
from typing import Optional
from torchrl.data import BoundedTensorSpec, CompositeSpec, UnboundedContinuousTensorSpec
from torchrl.envs import (
    CatTensors,
    EnvBase,
    Transform,
    TransformedEnv,
    UnsqueezeTransform,
)
from torchrl.envs.transforms.transforms import _apply_to_composite
from torchrl.envs.utils import step_mdp
#---------------------------
from torchrl.envs.libs.gym import GymEnv
from torchrl.envs.utils import check_env_specs, ExplorationType, set_exploration_type
from torchrl.modules import ProbabilisticActor, TanhNormal, ValueOperator
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE
from tqdm import tqdm

In [2]:
"""
cd "C:\Austin\Self_Projects\Pokemon_Sim\pokemon-showdown"
node pokemon-showdown start --no-security
"""

'\ncd "C:\\Austin\\Self_Projects\\Pokemon_Sim\\pokemon-showdown"\nnode pokemon-showdown start --no-security\n'

In [3]:
#Hyperparams 
format="gen9ou"
device="cpu"

In [4]:
class SmogonEnv(SinglesEnv, EnvBase):

    def __init__(
        self,
        seed=None,

        #EnvBase Variables
        device="cpu",
        batch_size = [],

        #SinglesEnv Variables
        account_configuration1: Optional[AccountConfiguration] = None,
        account_configuration2: Optional[AccountConfiguration] = None,
        battle_format: str = "gen8randombattle",
        start_timer_on_battle_start: bool = True,
        strict = True,
        fake = True,

        #Custom Input
        team1: Optional[Union[str, Teambuilder]] = None,
        team2: Optional[Union[str, Teambuilder]] = None,
    ):
        #self.observation_spaces = {
        #    agent: Box(np.array([0]), np.array([1]), dtype=np.int64)
        #    for agent in self.possible_agents
        #}

        SinglesEnv.__init__(
            self,
            #DoublesEnv Variables
            account_configuration1=account_configuration1,
            account_configuration2=account_configuration2,
            battle_format=battle_format,
            start_timer_on_battle_start=start_timer_on_battle_start,
            strict=strict,
            fake=fake,
            log_level=25
        )
        EnvBase.__init__(
            self,
            #EnvBase Variables
            device=device, 
            batch_size=batch_size
        )

        self.agent1.teampreview = self.teampreview
        self.agent2.teampreview = self.teampreview

        self.agent1.update_team(team1)
        self.agent2.update_team(team2)

        if seed is None:
            seed = torch.empty((), dtype=torch.int64).random_().item()
        self.set_seed(seed)

    #EnvBase Required Functions:

    def _reset(self, tensordict): 
        print("In Reset")

        obs, _ = self.reset()
        print("observation")
        #print(obs)
        out = TensorDict(
            {
                "observations":obs
            },
            batch_size=tensordict.shape,
        )

        print("Out Reset")

        return out, self.agent1.battle, self.agent2.battle 
    
    def _step(self, tensordict):

        print("In Step")
        
        for i in self.agents:
            print(i)
            print(tensordict["actions", i])
        
        
        observations, reward, terminated, truncated, _ = self.step(tensordict["actions"])

        out = TensorDict(
            {
                "observations":observations,
                "reward":reward,
                "terminated":terminated,
                "truncated":truncated,
            },
            batch_size=tensordict.shape,
        )

        print("Out Step")
        print(out)

        return out, self.agent1.battle, self.agent2.battle 
        

    def _set_seed(self, seed: Optional[int]): #Will write a custom reward function as required
        rng = torch.manual_seed(seed)
        self.rng = rng

    #DoublesEnv Required Functions:

    def calc_reward(self, battle) -> float:
        #Initially using built in reward helper function provided by Poke-Env
        #For simplicity of initiall implementation and testing

        return self.reward_computing_helper(
            battle, fainted_value=2.0, hp_value=1.0, victory_value=30.0
        )


    def embed_battle(self, battle: Battle): #Will write a custom reward function as required - __NEED TO UPDATE FOR DOUBLE BATTLE__
        assert isinstance(battle, Battle)
        # -1 indicates that the move does not have a base power
        # or is not available
        moves_base_power = -np.ones(4)
        moves_dmg_multiplier = np.ones(4)
        for i, move in enumerate(battle.available_moves):
            moves_base_power[i] = (
                move.base_power / 100
            )  # Simple rescaling to facilitate learning
            if battle.opponent_active_pokemon is not None:
                moves_dmg_multiplier[i] = move.type.damage_multiplier(
                    battle.opponent_active_pokemon.type_1,
                    battle.opponent_active_pokemon.type_2,
                    type_chart=battle.opponent_active_pokemon._data.type_chart,
                )

        # We count how many pokemons have fainted in each team
        fainted_mon_team = len([mon for mon in battle.team.values() if mon.fainted]) / 6
        fainted_mon_opponent = (
            len([mon for mon in battle.opponent_team.values() if mon.fainted]) / 6
        )

        # Final vector with 10 components
        final_vector = np.concatenate(
            [
                moves_base_power,
                moves_dmg_multiplier,
                [fainted_mon_team, fainted_mon_opponent],
            ]
        )
        return torch.Tensor(final_vector)

        
    
    def teampreview(self, battle: Battle) -> str: #Will write a custom reward function as required
        members = [1,2,3,4,5,6]#list(range(1, 7))
        random.shuffle(members)
        team_string = "/team " + "".join([str(x) for x in members])
        print(team_string)
        return team_string

    #Helper Functions:

    def print_teams(self):
        print(self.agent1._team.yield_team())
        print(self.agent2._team.yield_team())

    def print_torchrl_env_stats(self):
        print(self.device)
        print(self.batch_size)

    def select_game_team(self):
        pass

    def set_battle(self):
        self.battle1._finished=False
        self.battle2._finished=False

In [5]:
team1 = """
Meruem (Kingambit) @ Leftovers  
Ability: Supreme Overlord  
Tera Type: Ghost  
EVs: 164 HP / 252 Atk / 92 Spe  
Adamant Nature  
- Sucker Punch  
- Iron Head  
- Kowtow Cleave  
- Swords Dance  

Moebius (Deoxys-Speed) @ Life Orb  
Ability: Pressure  
Tera Type: Psychic  
EVs: 28 HP / 252 SpA / 228 Spe  
Modest Nature  
IVs: 0 Atk  
- Nasty Plot  
- Focus Blast  
- Psycho Boost  
- Shadow Ball  

Anomaly (Great Tusk) @ Booster Energy  
Ability: Protosynthesis  
Tera Type: Steel  
EVs: 252 Atk / 4 Def / 252 Spe  
Jolly Nature  
- Headlong Rush  
- Ice Spinner  
- Rapid Spin  
- Close Combat  

Yoshi (Dragonite) @ Choice Band  
Ability: Multiscale  
Shiny: Yes  
Tera Type: Normal  
EVs: 16 HP / 252 Atk / 240 Spe  
Adamant Nature  
- Outrage  
- Extreme Speed  
- Ice Spinner  
- Fire Punch  

Anomаly (Iron Moth) @ Booster Energy  
Ability: Quark Drive  
Tera Type: Ground  
EVs: 124 Def / 132 SpA / 252 Spe  
Timid Nature  
- Fiery Dance  
- Sludge Wave  
- Tera Blast  
- Toxic Spikes  

Siren (Primarina) @ Assault Vest  
Ability: Torrent  
Tera Type: Poison  
EVs: 76 HP / 252 SpA / 180 Spe  
Modest Nature  
IVs: 0 Atk  
- Surf  
- Moonblast  
- Whirlpool  
- Psychic Noise  
"""

In [6]:
team2 = """
Torkoal @ Heat Rock  
Ability: Drought  
Tera Type: Ground  
EVs: 104 HP / 252 SpA / 152 SpD  
Quiet Nature  
- Eruption  
- Overheat  
- Earthquake  
- Stealth Rock  

Hatterene @ Air Balloon  
Ability: Magic Bounce  
Tera Type: Ghost  
EVs: 252 HP / 252 Def / 4 SpD  
Relaxed Nature  
IVs: 0 Atk / 0 Spe  
- Trick Room  
- Psychic Noise  
- Dazzling Gleam  
- Healing Wish  

Raging Bolt @ Life Orb  
Ability: Protosynthesis  
Tera Type: Ghost  
EVs: 36 Def / 252 SpA / 220 Spe  
Modest Nature  
IVs: 20 Atk  
- Thunderclap  
- Weather Ball  
- Dragon Pulse  
- Solar Beam  

Slither Wing @ Assault Vest  
Ability: Protosynthesis  
Tera Type: Fire  
EVs: 40 HP / 252 Atk / 216 Spe  
Adamant Nature  
- U-turn  
- First Impression  
- Earthquake  
- Temper Flare  

Venusaur @ Life Orb  
Ability: Chlorophyll  
Tera Type: Fire  
EVs: 4 Atk / 252 SpA / 252 Spe  
Naive Nature  
- Growth  
- Giga Drain  
- Weather Ball  
- Earthquake  

Walking Wake @ Wise Glasses  
Ability: Protosynthesis  
Tera Type: Fairy  
EVs: 12 HP / 244 SpA / 252 Spe  
Timid Nature  
- Hydro Steam  
- Weather Ball  
- Draco Meteor  
- Flip Turn  

"""

In [7]:
env = SmogonEnv(device = device, battle_format=format, strict=True, fake = False, team1=team1, team2=team2, start_timer_on_battle_start=True)

/team 253614
/team 534612


In [8]:
env.rollout(1)

AttributeError: 'tuple' object has no attribute 'update'

In [12]:
battle_tracking = {"agent_1":[], "agent_2":[]}

In [12]:
import copy
td = TensorDict({},[],)
td, o1, o2 = env._reset(td)

env.agent1.battle_against(env.agent2)
battle_tracking["agent_1"].append(copy.deepcopy(o1))
battle_tracking["agent_2"].append(copy.deepcopy(o2))

print(env.agents[0])
print(env.agents[1])
print(env.battle1.finished)

In Reset
observation
Out Reset
SmogonEnv 86ele
SmogonEnv dafna
False


In [ ]:

import copy
td = TensorDict({
        "actions":{
                env.agents[0]: (
                    torch.Tensor([5], device=env.device).type(torch.int64)
                ),

                env.agents[1]: (
                    torch.Tensor([5], device=env.device).type(torch.int64)
                )
            }})

out, o1, o2 = env._step(td)
battle_tracking["agent_1"].append(copy.deepcopy(o1))
battle_tracking["agent_2"].append(copy.deepcopy(o1))


In Step
SmogonEnv v90kp
tensor([5])
SmogonEnv jxnso
tensor([5])


In [14]:
out["observations"]["SmogonEnv 8fufq"]

tensor([-1.0000, -1.0000, -1.0000, -1.0000,  1.0000,  1.0000,  1.0000,  1.0000,
         0.0000,  0.1667])

In [36]:
import pickle
with open('example_battle_dict.pickle', 'wb') as handle:
    pickle.dump(battle_tracking, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [3]:
with open('example_battle_dict.pickle', 'rb') as handle:
    battle_tracking = pickle.load(handle)

In [24]:
battle_tracking["agent_1"][-4].all_active_pokemons

[slitherwing (pokemon object) [Active: True, Status: FNT],
 greattusk (pokemon object) [Active: True, Status: None]]